In [1]:
import pandas as pd

rankings_df = pd.read_csv("/scicore/home/schwede/barta0000/project/data/processed/evaluation/current/rankings.csv")
df = pd.read_csv("/scicore/home/schwede/barta0000/project/data/raw/current/eval_aligned.csv")
design_df = pd.read_csv("/scicore/home/schwede/barta0000/project/data/raw/snir/eval_aligned.csv")

missing = set(rankings_df["metric"]) - set(design_df.columns)

print("Missing metrics:")
print(missing)


Missing metrics:
set()


In [2]:
import yaml
import pandas as pd

with open("/scicore/home/schwede/barta0000/project/config/metric_data.yaml") as f:
    metrics_config = yaml.safe_load(f)

metrics_config = metrics_config["metrics"]

In [3]:
def select_top_metrics(rankings_df, n=10):
    """
    Select globally best metrics by PR-AUC. Top n.
    """
    top = (
        rankings_df
        .sort_values("pr_auc", ascending=False)
        .head(n)
        .copy()
    )

    return top

top10_global = select_top_metrics(rankings_df, n=10)

print(top10_global[["metric", "pr_auc"]])

selected_metrics_global = top10_global["metric"].tolist()
directions_global = dict(zip(top10_global["metric"], top10_global["direction"]))

                               metric  pr_auc
0                          af3_iplddt  0.8395
1                     boltz_free_ipde  0.8383
2                            af3_iptm  0.8336
3                    boltz_free_ipsae  0.8323
4                             af3_pae  0.8310
5      boltz_free_per_chain_pair_iptm  0.8297
6                         af3_pdockq2  0.8288
7              boltz_template_pdockq2  0.8285
8  boltz_template_per_chain_pair_iptm  0.8257
9                            cf_ipsae  0.8230


In [4]:
def add_metric_metadata(rankings_df, config):

    meta = (
        pd.DataFrame(config)
        .T
        .reset_index()
        .rename(columns={"index": "metric"})
    )

    # direction already exists in rankings.csv
    meta = meta.drop(columns=["direction"], errors="ignore")

    return rankings_df.merge(
        meta,
        on="metric",
        how="left"
    )

rankings_meta = add_metric_metadata(rankings_df,metrics_config)

In [5]:
print(metrics_config.keys())

dict_keys(['af3_ipae', 'af3_iplddt', 'af3_ipsae', 'af3_iptm', 'af3_rank0_masif', 'af3_pdockq', 'af3_pdockq2', 'af3_lis', 'cf_ipae', 'cf_iplddt', 'cf_ipsae', 'cf_iptm', 'cf_rank0_masif', 'pred_ilddt', 'cf_pdockq', 'cf_pdockq2', 'cf_lis', 'esmfold_ipae', 'esmfold_iplddt', 'chai_unconstrained_iptm', 'chai_unconstrained_per_chain_ab_iptm_mean', 'chai_unconstrained_interface_iptm', 'chai_unconstrained_ipsae', 'chai_unconstrained_pdockq', 'chai_unconstrained_pdockq2', 'chai_unconstrained_lis', 'chai_unconstrained_per_chain_pair_iptm', 'chai_constrained_iptm', 'chai_constrained_per_chain_ab_iptm_mean', 'chai_constrained_interface_iptm', 'chai_constrained_ipsae', 'chai_constrained_pdockq', 'chai_constrained_pdockq2', 'chai_constrained_lis', 'chai_constrained_per_chain_pair_iptm', 'boltz_free_iptm', 'boltz_free_iplddt', 'boltz_free_ipde', 'boltz_free_per_chain_pair_iptm', 'boltz_free_ipsae', 'boltz_free_pdockq', 'boltz_free_pdockq2', 'boltz_free_lis', 'boltz_template_iptm', 'boltz_template_ipld

In [6]:
def select_diverse_metrics(rankings_meta, n=10):
    """Select best metric from each family,then take the best n families."""

    best_per_family = (
        rankings_meta
        .sort_values("pr_auc", ascending=False)
        .groupby("family", as_index=False)
        .first()
    )

    selected = (
        best_per_family
        .sort_values("pr_auc", ascending=False)
        .head(n)
    )

    return selected

top10_diverse = select_diverse_metrics(rankings_meta,n=10)



print(top10_diverse[["metric", "family", "category", "pr_auc"]])

                            metric               family    category  pr_auc
3                       af3_iplddt               iplddt   interface  0.8395
2                  boltz_free_ipde                 ipde   interface  0.8383
5                         af3_iptm                 iptm   interface  0.8336
4                 boltz_free_ipsae                ipsae   interface  0.8323
7                          af3_pae                  pae  confidence  0.8310
10  boltz_free_per_chain_pair_iptm  per_chain_pair_iptm   interface  0.8297
9                      af3_pdockq2              pdockq2   interface  0.8288
12                         af3_ptm                  ptm  confidence  0.8222
6                   boltz_free_lis                  lis   interface  0.8212
0      boltz_free_confidence_score     confidence_score  confidence  0.8139


In [11]:
import numpy as np
def consensus_rank(design_df, selected_metrics, directions, agg="mean"):
    """Rank designs by consensus across selected metrics. Aggregated by mean -> arithmetic mean of ranks or product -> geometric mean of ranks in log space. Lower consensus_score = better in both cases."""

    ranked = design_df.copy()
    rank_cols = []

    for metric in selected_metrics:
        col = f"{metric}_rank"
        ascending = directions[metric] == 1          # dir == 1: higher is better --> df direction-aligned

        #ranked[col] = pd.qcut(ranked[metric].rank(ascending=ascending), q=20, labels=False, duplicates="drop") + 1  # ranked based on 20 bin of 5%
        ranked[col] = ranked[metric].rank(ascending=ascending, pct=True)   # 0=best … 1=worst   # percentile-based
        rank_cols.append(col)

    if agg == "mean":
        ranked["consensus_score"] = ranked[rank_cols].mean(axis=1)

    elif agg == "product":
        # geometric mean via mean of logs: stable, and comparable across
        # rows even if some metrics are NaN
        ranked["consensus_score"] = np.log(ranked[rank_cols]).mean(axis=1)
    else:
        raise ValueError(f"agg must be 'mean' or 'product', got {agg!r}")

    return ranked.sort_values("consensus_score", ascending=True)

#ranked_global = consensus_rank(design_df, selected_metrics_exp1, directions_exp1, agg="product")  # exp1: globally best metrics
ranked_fam = consensus_rank(design_df, top10_diverse["metric"].tolist(), directions_diverse, agg="product") # exp2: best per family 

#ranked_global.head(10)
ranked_fam.head(10)

,source,type,binder,sample,af3_pae,af3_plddt,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq2,...,boltz_free_ipde_rank,af3_iptm_rank,boltz_free_ipsae_rank,af3_pae_rank,boltz_free_per_chain_pair_iptm_rank,af3_pdockq2_rank,af3_ptm_rank,boltz_free_lis_rank,boltz_free_confidence_score_rank,consensus_score
27,snir_ab,alanine_scan,0,5ijk_acx_cut_all_A,-4.276973,90.392437,-4.580952,71.830438,0.058250,0.08885,...,1.000000,0.171429,0.028571,1.000000,0.028571,0.142857,0.028571,0.028571,0.028571,-2.309568
23,snir_ab,alanine_scan,0,5ijk_acx_cut_A_I114A,-3.581104,91.142842,-3.039530,67.753721,0.024187,0.06940,...,0.914286,0.100000,0.057143,0.914286,0.057143,0.057143,0.085714,0.085714,0.085714,-2.060767
25,snir_ab,alanine_scan,0,5ijk_acx_cut_A_R55A,-3.411108,91.770396,-3.270139,64.294001,0.024243,0.08470,...,0.971429,0.100000,0.085714,0.800000,0.085714,0.114286,0.200000,0.057143,0.114286,-1.983328
24,snir_ab,alanine_scan,0,5ijk_acx_cut_A_K57A,-3.333153,92.231214,-3.738864,66.901441,0.023776,0.08380,...,0.857143,0.028571,0.228571,0.742857,0.257143,0.085714,0.200000,0.142857,0.228571,-1.678551
32,snir_ab,alanine_scan,0,5ijk_acx_cut_B_Y108A,-3.341985,91.641826,-3.902778,65.016859,0.024165,0.06370,...,0.657143,0.142857,0.400000,0.771429,0.371429,0.028571,0.200000,0.171429,0.285714,-1.557530
8,snir_ab,alanine_scan,0,5ifj_abc_cut_all_A,-3.597543,93.235998,-2.204441,83.750294,0.189334,0.33130,...,0.942857,0.428571,0.142857,0.942857,0.142857,0.428571,0.114286,0.228571,0.057143,-1.305856
16,snir_ab,alanine_scan,0,5ig7_abc_cut_all_A,-3.805987,92.210506,-2.444928,76.576550,0.061503,0.17770,...,0.828571,0.200000,0.200000,0.971429,0.171429,0.285714,0.057143,0.285714,0.514286,-1.230265
31,snir_ab,alanine_scan,0,5ijk_acx_cut_B_Y107A,-3.163120,93.553850,-2.214583,72.905361,0.026280,0.12810,...,0.485714,0.057143,0.628571,0.685714,0.657143,0.200000,0.300000,0.314286,0.314286,-1.133223
26,snir_ab,alanine_scan,0,5ijk_acx_cut_A_T113A,-2.042686,97.319849,-1.473958,97.583035,0.760557,0.93485,...,0.885714,0.971429,0.171429,0.028571,0.200000,1.000000,1.000000,0.114286,0.171429,-1.101137
22,snir_ab,alanine_scan,0,5ijk_acx_cut_A_I112A,-2.957804,94.094014,-2.786161,76.558583,0.112243,0.25410,...,0.542857,0.400000,0.314286,0.542857,0.342857,0.342857,0.442857,0.200000,0.142857,-1.096375


In [13]:
# old: ranks samples one-by-one per metric

def consensus_rank(design_df, selected_metrics, directions):
    """Scores each design by agreement across selected metrics. For each selected metric, it ranks all rows in design_df on that metric,
    orienting so rank 1 = best. If direction == 1 it ranks descending; otherwise ascending. Consensus_score is the mean of those per-metric ranks for each row."""

    ranked = design_df.copy()

    rank_cols = []

    for metric in selected_metrics:

        col = f"{metric}_rank"

        if directions[metric] == 1:
            ranked[col] = ranked[metric].rank(ascending=False)

        else:
            ranked[col] = ranked[metric].rank(ascending=True)

        rank_cols.append(col)

    ranked["consensus_score"] = (ranked[rank_cols].mean(axis=1))

    return ranked.sort_values("consensus_score", ascending=True)

# experiment 1
#ranked_global = consensus_rank(design_df, selected_metrics_exp1, directions_exp1)

# experiment 2
ranked_fam = consensus_rank(design_df,top10_diverse["metric"].tolist(), dict(zip(top10_diverse["metric"], top10_diverse["direction"])))

#ranked_global.head(10)
ranked_fam.head(10)


,source,type,binder,sample,af3_pae,af3_plddt,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq2,...,boltz_free_ipde_rank,af3_iptm_rank,boltz_free_ipsae_rank,af3_pae_rank,boltz_free_per_chain_pair_iptm_rank,af3_pdockq2_rank,af3_ptm_rank,boltz_free_lis_rank,boltz_free_confidence_score_rank,consensus_score
17,snir_ab,alanine_scan,0,5ig7_abc_cut_B_G114A,-2.436673,95.869087,-1.884375,95.531792,0.810392,0.90170,...,33.0,2.0,2.0,33.0,2.0,2.0,2.0,3.0,4.0,8.60
1,snir_ab,original,1,5ig7_abc_cut,-2.572969,95.599717,-2.020938,94.909961,0.785245,0.89390,...,34.0,4.5,4.0,29.0,4.0,3.0,8.0,1.0,1.0,9.45
15,snir_ab,alanine_scan,0,5ig7_abc_cut_A_Y66A,-2.572738,95.767986,-1.959615,95.063667,0.786285,0.88890,...,28.0,8.0,7.0,30.0,7.0,4.0,8.0,5.0,2.0,10.30
13,snir_ab,alanine_scan,0,5ig7_abc_cut_A_K110A,-2.806122,94.950463,-2.220000,93.355815,0.756785,0.85780,...,35.0,11.5,1.0,21.0,1.0,12.0,15.5,2.0,3.0,11.90
0,snir_ab,original,1,5ifj_abc_cut,-2.593883,95.404666,-1.993750,95.036725,0.788184,0.88085,...,21.0,2.0,15.0,28.0,15.0,8.0,8.0,8.0,9.0,11.90
4,snir_ab,alanine_scan,0,5ifj_abc_cut_A_K110A,-2.790728,94.871984,-2.275417,92.998114,0.769552,0.85065,...,31.0,4.5,3.0,22.0,3.0,14.0,15.5,4.0,11.0,12.60
5,snir_ab,alanine_scan,0,5ifj_abc_cut_A_S64A,-2.539446,95.613237,-2.072188,94.813927,0.778962,0.88450,...,11.0,8.0,17.0,32.0,17.0,6.0,8.0,11.0,13.0,13.00
3,snir_ab,alanine_scan,0,5ifj_abc_cut_A_D111A,-2.815847,95.282854,-2.100000,94.730775,0.765295,0.88195,...,24.0,8.0,12.0,19.0,12.0,7.0,15.5,7.0,20.0,13.35
10,snir_ab,alanine_scan,0,5ifj_abc_cut_B_V38A,-2.714935,95.220947,-2.254062,93.465938,0.758796,0.85940,...,23.0,8.0,13.0,24.0,14.0,11.0,8.0,6.0,17.0,13.80
18,snir_ab,alanine_scan,0,5ig7_abc_cut_B_L116A,-2.697161,95.298420,-2.129575,93.730328,0.756877,0.86445,...,22.0,11.5,19.0,26.0,19.0,9.0,8.0,9.0,5.0,14.05
